# 04 · Leave-one-centre-out external validation & MGMT probe

**Reviewer 3.5** — genuine cross-site generalisation: for each centre, train on the
other three (patient-disjoint) and test on the held-out centre.
**Reviewer 4.4** — exploratory MGMT prediction from the learned subtype scores.

Runtime: ~10–15 min on CPU (four trainings).

In [ ]:
# --- notebook preamble: locate the reusable analysis library ------------------
import os, sys, warnings
warnings.filterwarnings("ignore")
LIB = os.path.join(os.path.dirname(os.path.abspath("__file__")), "lib")
if not os.path.isdir(LIB):
    LIB = os.path.join(os.getcwd(), "lib")
sys.path.insert(0, LIB)

# _lib_raman defines ROOT / DATA / MODEL_PATH / META_PATH (absolute paths to the
# original project). If you moved the data, override them here, e.g.:
#   import _lib_raman as L; L.ROOT = "/path/to/Project_Raman"; L.DATA = ...
import _lib_raman as L
print("Project ROOT :", L.ROOT)
print("Graph DATA   :", L.DATA)
print("Model        :", L.MODEL_PATH)
print("Classes      :", L.CLASSES)

import _lib_train as T
import numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
params = torch.load(L.MODEL_PATH, weights_only=False, map_location='cpu')['params']

## 1 · Leave-one-centre-out (LOCO)

In [ ]:
graphs = T.assemble_graphs(hop=3, graph_set='small')
centres = sorted({g.center for g in graphs if g.center is not None})
rows=[]
for held in centres:
    tr=[g for g in graphs if g.center not in (held,None)]
    te=[g for g in graphs if g.center==held]
    model,_=T.train_model(tr,te,'cpu',params,epochs=120,patience=15,bias=True,log=lambda *a:None)
    pat=T.aggregate_patient(T.predict(model,te,'cpu'))
    a={c:(roc_auc_score((pat.y_true==ci).astype(int),pat[f'score_{c}']) if pat.y_true.nunique()>1 and (pat.y_true==ci).sum()>0 else np.nan)
       for ci,c in enumerate(L.CLASSES)}
    rows.append(dict(held_out=held,n_pat=pat.patient_id.nunique(),mean_auc=np.nanmean(list(a.values())),
                     **{f'auc_{c}':a[c] for c in L.CLASSES}))
    print(f'{held}: patient mean AUC {rows[-1]["mean_auc"]:.3f}')
loco=pd.DataFrame(rows); loco.round(3)

RTK2 typically generalises best across centres; rare classes are unstable given few
positive patients per site.

## 2 · Exploratory MGMT probe (patient-level 5-fold CV)

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
mp = pd.read_csv('outputs/master_predictions_meta.csv')
v = mp[(mp.graph_set=='small')&(mp.hop==3)]
sc=[f'logit_{c}' for c in L.CLASSES]
pat=v.groupby('patient_id').agg({**{c:'mean' for c in sc},'MGMTStatus':'first'}).reset_index().dropna(subset=['MGMTStatus'])
X=StandardScaler().fit_transform(pat[sc].values); y=(pat.MGMTStatus.str.lower()=='methylated').astype(int).values
oof=np.zeros(len(y))
for tri,tei in StratifiedKFold(5,shuffle=True,random_state=42).split(X,y):
    oof[tei]=LogisticRegression(max_iter=1000).fit(X[tri],y[tri]).predict_proba(X[tei])[:,1]
print(f'MGMT CV AUC = {roc_auc_score(y,oof):.3f}  (n={len(y)}, methylated={y.sum()})  — exploratory only')